# Mastering Data Formatting with `format.py`

This notebook provides a comprehensive tutorial on using the `format.py` Python module. This module offers utilities for converting conversation transcripts and speech recognition responses between various formats, specifically targeting Google Contact Center AI Insights and Google Cloud Data Loss Prevention (DLP).

Whether you're dealing with AWS transcripts, Genesys Cloud data, or Google Cloud Speech-to-Text outputs, `format.py` aims to streamline your data preparation workflow.

## Table of Contents
1.  Setup and Module Installation
2.  Understanding the `Role` Enum
3.  The `Dlp` Class (Data Loss Prevention)
4.  The `Insights` Class (Conversation Formatting)
    *   Converting from AWS Transcripts (`from_aws`)
    *   Converting from Genesys Cloud Transcripts (`from_genesys_cloud`)
5.  The `Speech` Class (Speech-to-Text Response Handling)
    *   Converting Speech v1 Recognizer Responses
    *   Converting Speech v2 Recognizer Responses
    *   Extracting Text from Speech v2 JSON (`v2_json_to_dict`)


## 1. Setup and Module Installation

First, we'll install necessary libraries like `jsonschema` (used for validating input transcripts) and `google-cloud-speech` (required for type hints, though we'll mock its responses for demonstration).

Next, we'll make the `format.py` module available to our notebook environment. We'll write its content to a file, create necessary schema directories and files, and then import it.

In [ ]:
# Install necessary libraries
!pip install jsonschema google-cloud-speech protobuf==3.20.0 --quiet

In [ ]:
# Create the format.py module and its dependencies in the current directory
import os
import json
from pathlib import Path

# Define the content of format.py
format_py_content = """
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""A class for all Insigths things related"""

import json
import os
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Optional
# pyright: reportMissingModuleSource=false
import jsonschema
from google.cloud.speech_v1 import types as types_v1
from google.cloud.speech_v2 import types as types_v2


class Role(Enum):
    """An enum to represent the role of a speaker in a conversation."""

    AGENT = "AGENT"
    CUSTOMER = "CUSTOMER"


_MICROSECONDS_PER_SECOND = 1000000


class Dlp:
    """Class to convert from DLP to Insights and from Insights to DLP"""

    def __init__(self):
        pass

    def from_conversation_json(self):
        "Converts from Conversation JSON to DLP Table"
        ## https://cloud.google.com/contact-center/insights/docs/conversation-data-format
        raise NotImplementedError

    def from_recognize_response(self):
        "Converts from Recognize Response to DLP Table"
        ## https://cloud.google.com/python/docs/reference/speech/latest/
        ## google.cloud.speech_v1.types.RecognizeResponse
        raise NotImplementedError

    def redact(self):
        """Redacts the contents from a DLP table"""
        raise NotImplementedError


class Insights:
    """Class to convert from other common formats to conversation_Json"""

    def __init__(self):
        pass

    def from_aws(
        self, transcript: dict, transcript_datetime_string: Optional[str] = None
    ) -> dict:
        """Converts transcripts from AWS to Insights JsonConversationInput format

        Args:
            transcript: AWS transcript to convert.

        Returns:
            Insights JsonConversationInput format
        """
        # AWS transcripts provides time as offset. Does not contain date and time information.
        # So we take the current epoch at the time of import and then convert to insights
        # compatible format

        # validate transcript_datetime format
        if transcript_datetime_string:
            datetime_object = datetime.strptime(
                transcript_datetime_string, "%Y/%m/%d %H:%M:%S"
            )
            transcript_timestamp = int(datetime_object.timestamp())
        else:
            current_datetime = datetime.now()
            transcript_timestamp = int(current_datetime.timestamp())

        # load aws transcript schema
        file_path = os.path.join(
            os.path.dirname(__file__), "utils/schemas", "aws_schema.json"
        )
        with open(file_path, "r", encoding="utf-8") as file:
            aws_schema = json.load(file)

        # validate the transcript with aws_schema
        jsonschema.validate(instance=transcript, schema=aws_schema)

        insights_transcript: Dict[str, list[dict]] = {"entries": []}

        for entry in transcript["Transcript"]:
            user_id = 1 if entry["ParticipantId"] == "AGENT" else 2
            ent = {
                "role": entry["ParticipantId"],
                "start_timestamp_usec": int(
                    (entry["BeginOffsetMillis"] / 1000) + transcript_timestamp
                )
                * _MICROSECONDS_PER_SECOND,
                "text": entry["Content"],
                "user_id": user_id,
            }
            insights_transcript["entries"].append(ent)

        return insights_transcript

    def from_genesys_cloud(self, transcript: dict) -> dict:
        """Converts transcripts from Genesys Cloud to conversation_json

        Args:
            transcript: Genesys Cloud transcript to convert.

        Returns:
            Insights JsonConversationInput format
        """

        # load genesys_cloud transcript schema
        file_path = os.path.join(
            os.path.dirname(__file__), "utils/schemas", "genesys_cloud_schema.json"
        )
        with open(file_path, "r", encoding="utf-8") as file:
            genesys_cloud_schema = json.load(file)

        # validate the transcript with genesys_cloud_schema
        jsonschema.validate(instance=transcript, schema=genesys_cloud_schema)

        insights_transcript: Dict[str, list[dict]] = {"entries": []}

        for entry in transcript["transcripts"][0]["phrases"]:
            role = (
                Role.CUSTOMER
                if entry["participantPurpose"] == "external"
                else Role.AGENT
            )
            user_id = 1 if role == Role.AGENT else 2

            ent = {
                "role": role.value,
                # Genesys Cloud contains the timestamps in milliseconds of epochtime.
                # Since Insights need the timestamps in microseconds, multiply by 1000
                "start_timestamp_usec": entry["startTimeMs"] * 1000,
                "text": entry["text"],
                "user_id": user_id,
            }
            insights_transcript["entries"].append(ent)

        return insights_transcript

    def from_insights_bq(self):
        """Converts transcripts from Insights BQ format to conversation_json"""
        raise NotImplementedError

    def from_verint(self):
        """Converts transcripts from Verint to conversation_json"""
        raise NotImplementedError

    def from_nice(self):
        """Converts transcripts from NICE to conversation_json"""
        raise NotImplementedError

    def from_dlp_table(self):
        """Converts transcripts from a DLP table to conversation_json"""
        raise NotImplementedError


class Speech:
    """Class to convert speech things to other things"""

    def __init__(self):
        pass

    def v1_recognizer_to_dict(
        self, recognizer_response: types_v1.LongRunningRecognizeResponse
    ) -> list[str]:
        """Converts Speech v1 return to list[str]"""
        dict_response = json.loads(
            type(recognizer_response).to_json(recognizer_response)
        )
        return dict_response

    def v1_recognizer_to_json(
        self, recognizer_response: types_v1.LongRunningRecognizeResponse
    ) -> str:
        """Converts Speech v1 return to str(json)"""
        dict_response = json.loads(
            type(recognizer_response).to_json(recognizer_response)
        )
        return json.dumps(dict_response)

    def v2_recognizer_to_json(
        self, recognizer_response: types_v2.cloud_speech.RecognizeResponse
    ) -> str:
        """Converts Speech v1 return to str(json)"""
        dict_response = json.loads(
            type(recognizer_response).to_json(recognizer_response)
        )
        return json.dumps(dict_response)

    def v2_recognizer_to_dict(
        self, recognizer_response: types_v2.cloud_speech.RecognizeResponse
    ) -> list[str]:
        """Converts Speech v1 return to list[str]"""
        dict_response = json.loads(
            type(recognizer_response).to_json(recognizer_response)
        )
        return dict_response

    def v2_json_to_dict(
        self,
        v2_json: dict,
    ) -> Dict[str, List[Dict[str, Any]]]:
        """Converts speech v2 json dict to a dict of results"""
        rr_format_data: Dict[str, List[Dict[str, Any]]] = {"results": []}
        uid = 0
        for item in v2_json["results"]:
            if "alternatives" not in item:
                continue
            for alternative in item["alternatives"]:
                if "transcript" not in alternative:
                    continue
                rr_format_data["results"].append(
                    {"uid": uid, "text": alternative["transcript"]}
                )
                uid += 1
        return rr_format_data
"""

# Write format.py to a file
with open("format_module.py", "w", encoding="utf-8") as f:
    f.write(format_py_content)

print("format_module.py created.")

# Create the 'utils/schemas' directory
schema_dir = Path("utils/schemas")
schema_dir.mkdir(parents=True, exist_ok=True)

# Define and write aws_schema.json
aws_schema_content = {
  "type": "object",
  "properties": {
    "Transcript": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "ParticipantId": {"type": "string", "enum": ["AGENT", "CUSTOMER"]},
          "BeginOffsetMillis": {"type": "integer"},
          "Content": {"type": "string"}
        },
        "required": ["ParticipantId", "BeginOffsetMillis", "Content"]
      }
    }
  },
  "required": ["Transcript"]
}
with open(schema_dir / "aws_schema.json", "w", encoding="utf-8") as f:
    json.dump(aws_schema_content, f, indent=2)
print("aws_schema.json created.")

# Define and write genesys_cloud_schema.json
genesys_cloud_schema_content = {
  "type": "object",
  "properties": {
    "transcripts": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "phrases": {
            "type": "array",
            "items": {
              "type": "object",
              "properties": {
                "participantPurpose": {"type": "string", "enum": ["internal", "external"]},
                "startTimeMs": {"type": "integer"},
                "text": {"type": "string"}
              },
              "required": ["participantPurpose", "startTimeMs", "text"]
            }
          }
        },
        "required": ["phrases"]
      }
    }
  },
  "required": ["transcripts"]
}
with open(schema_dir / "genesys_cloud_schema.json", "w", encoding="utf-8") as f:
    json.dump(genesys_cloud_schema_content, f, indent=2)
print("genesys_cloud_schema.json created.")

# Mock the Google Cloud Speech types for demonstration without needing actual API calls
from google.protobuf.json_format import MessageToJson
from unittest.mock import MagicMock

# Mock google.cloud.speech_v1.types.LongRunningRecognizeResponse
class MockV1RecognizeResponse:
    # For simplicity, we'll make to_json a static method that returns a fixed string.
    # In a real scenario, this would be a protobuf message instance method.
    @staticmethod
    def to_json(instance):
        return json.dumps({
            "results": [
                {"alternatives": [{"transcript": "hello world", "confidence": 0.95}]},
                {"alternatives": [{"transcript": "how are you doing today", "confidence": 0.90}]}
            ]
        })

# Mock google.cloud.speech_v2.types.cloud_speech.RecognizeResponse
class MockV2RecognizeResponse:
    @staticmethod
    def to_json(instance):
        return json.dumps({
            "results": [
                {"alternatives": [{"transcript": "good morning customer", "confidence": 0.98, "words": []}]},
                {"alternatives": [{"transcript": "how can I assist you", "confidence": 0.92, "words": []}]}
            ]
        })

# Replace the actual types with our mocks within a dummy module for import
import sys
if 'google.cloud.speech_v1.types' not in sys.modules:
    sys.modules['google.cloud.speech_v1.types'] = MagicMock()
sys.modules['google.cloud.speech_v1.types'].LongRunningRecognizeResponse = MockV1RecognizeResponse

if 'google.cloud.speech_v2.types' not in sys.modules:
    sys.modules['google.cloud.speech_v2.types'] = MagicMock()
if 'google.cloud.speech_v2.types.cloud_speech' not in sys.modules:
    sys.modules['google.cloud.speech_v2.types.cloud_speech'] = MagicMock()
sys.modules['google.cloud.speech_v2.types.cloud_speech'].RecognizeResponse = MockV2RecognizeResponse

print("Google Cloud Speech types mocked.")

# Now import the module
from format_module import Role, Dlp, Insights, Speech

print("format_module imported successfully!")


## 2. Understanding the `Role` Enum

The `Role` enum defines the participants in a conversation, typically `AGENT` and `CUSTOMER`. This ensures consistent categorization across different transcript formats.

In [ ]:
print(f"Agent Role: {Role.AGENT.value}")
print(f"Customer Role: {Role.CUSTOMER.value}")

# You can compare roles
if Role.AGENT == Role.AGENT:
    print("Roles can be compared.")

## 3. The `Dlp` Class (Data Loss Prevention)

The `Dlp` class is intended for converting data to and from a format compatible with Google Cloud Data Loss Prevention (DLP) and performing redaction. At the moment, all its methods are marked as `NotImplementedError`, indicating they are placeholders for future development.

In [ ]:
dlp_converter = Dlp()

try:
    dlp_converter.from_conversation_json()
except NotImplementedError as e:
    print(f"`from_conversation_json` method: {e}")

try:
    dlp_converter.redact()
except NotImplementedError as e:
    print(f"`redact` method: {e}")


## 4. The `Insights` Class (Conversation Formatting)

The `Insights` class specializes in converting transcripts from various external platforms into a unified `JsonConversationInput` format suitable for Google Contact Center AI Insights. This allows for standardized analysis regardless of the original data source.

### Converting from AWS Transcripts (`from_aws`)

The `from_aws` method takes an AWS transcript dictionary and converts it to the Insights format. AWS transcripts typically provide time as an offset from the start of the conversation. This method can optionally take a `transcript_datetime_string` to set the base timestamp for the conversation; otherwise, it uses the current time.

The method also validates the input `transcript` against a predefined `aws_schema.json`.

In [ ]:
insights_converter = Insights()

# Example AWS transcript data
aws_transcript_data = {
    "Transcript": [
        {
            "ParticipantId": "AGENT",
            "BeginOffsetMillis": 0,
            "Content": "Hello, thank you for calling. How can I help you today?"
        },
        {
            "ParticipantId": "CUSTOMER",
            "BeginOffsetMillis": 3500,
            "Content": "Hi, I'm having trouble with my internet connection."
        },
        {
            "ParticipantId": "AGENT",
            "BeginOffsetMillis": 7000,
            "Content": "I see. Let me check your account details."
        }
    ]
}

# Convert using from_aws (with an explicit start time)
transcript_datetime = "2023/10/26 09:30:00"
insights_aws_output = insights_converter.from_aws(
    aws_transcript_data, transcript_datetime_string=transcript_datetime
)

print("\n--- Converted AWS Transcript (with explicit datetime) ---")
print(json.dumps(insights_aws_output, indent=2))

# Convert using from_aws (using current time as base)
insights_aws_output_now = insights_converter.from_aws(
    aws_transcript_data
)

print("\n--- Converted AWS Transcript (using current datetime) ---")
print(json.dumps(insights_aws_output_now, indent=2))


# Example of schema validation failure (uncomment to test)
# invalid_aws_transcript_data = {
#     "Transcript": [
#         {"ParticipantId": "UNKNOWN", "BeginOffsetMillis": 0, "Content": "Invalid role."}
#     ]
# }
# try:
#     insights_converter.from_aws(invalid_aws_transcript_data)
# except jsonschema.exceptions.ValidationError as e:
#     print(f"\nSchema validation failed as expected: {e.message}")


### Converting from Genesys Cloud Transcripts (`from_genesys_cloud`)

The `from_genesys_cloud` method transforms transcripts from Genesys Cloud into the Insights `JsonConversationInput` format. Genesys Cloud transcripts provide timestamps in milliseconds of epoch time, which this method correctly converts to microseconds as required by Insights.

Similar to `from_aws`, it validates the input against a `genesys_cloud_schema.json`.

In [ ]:
# Example Genesys Cloud transcript data
genesys_cloud_transcript_data = {
    "transcripts": [
        {
            "phrases": [
                {
                    "participantPurpose": "internal", # AGENT
                    "startTimeMs": 1678886400000, # Example epoch time in ms
                    "text": "Welcome to our support line."
                },
                {
                    "participantPurpose": "external", # CUSTOMER
                    "startTimeMs": 1678886402500,
                    "text": "I need help resetting my password."
                },
                {
                    "participantPurpose": "internal",
                    "startTimeMs": 1678886405100,
                    "text": "No problem, I can help you with that."
                }
            ]
        }
    ]
}

# Convert using from_genesys_cloud
insights_genesys_output = insights_converter.from_genesys_cloud(genesys_cloud_transcript_data)

print("--- Converted Genesys Cloud Transcript ---")
print(json.dumps(insights_genesys_output, indent=2))

# Example of schema validation failure (uncomment to test)
# invalid_genesys_cloud_transcript_data = {
#     "transcripts": [
#         {
#             "phrases": [
#                 {"participantPurpose": "invalid_purpose", "startTimeMs": 0, "text": "Invalid data."}
#             ]
#         }
#     ]
# }
# try:
#     insights_converter.from_genesys_cloud(invalid_genesys_cloud_transcript_data)
# except jsonschema.exceptions.ValidationError as e:
#     print(f"\nSchema validation failed as expected: {e.message}")


## 5. The `Speech` Class (Speech-to-Text Response Handling)

The `Speech` class provides utilities for processing responses from Google Cloud Speech-to-Text API, specifically for v1 (LongRunningRecognizeResponse) and v2 (RecognizeResponse) models. It allows converting these complex Protobuf responses into more manageable dictionary or JSON string formats, and also offers a helper to extract just the transcript text.

In [ ]:
speech_converter = Speech()

# Create mock response objects using our defined mock classes
mock_v1_response_obj = MockV1RecognizeResponse()
mock_v2_response_obj = MockV2RecognizeResponse()

print("--- Speech v1 Recognizer to Dictionary ---")
v1_dict = speech_converter.v1_recognizer_to_dict(mock_v1_response_obj)
print(json.dumps(v1_dict, indent=2))

print("\n--- Speech v1 Recognizer to JSON String ---")
v1_json_str = speech_converter.v1_recognizer_to_json(mock_v1_response_obj)
print(v1_json_str)


### Converting Speech v2 Recognizer Responses

Similar to v1, this section demonstrates how to convert Speech v2 `RecognizeResponse` objects into dictionary or JSON string formats.

In [ ]:
print("--- Speech v2 Recognizer to Dictionary ---")
v2_dict = speech_converter.v2_recognizer_to_dict(mock_v2_response_obj)
print(json.dumps(v2_dict, indent=2))

print("\n--- Speech v2 Recognizer to JSON String ---")
v2_json_str = speech_converter.v2_recognizer_to_json(mock_v2_response_obj)
print(v2_json_str)


### Extracting Text from Speech v2 JSON (`v2_json_to_dict`)

The `v2_json_to_dict` method is a convenience function to parse a Speech v2 JSON dictionary (typically obtained from a `RecognizeResponse`) and extract just the recognized transcripts into a simpler dictionary format, assigning a unique ID to each utterance.

In [ ]:
# Example Speech v2 JSON dictionary (could be from json.loads(v2_json_str))
sample_v2_json = {
    "results": [
        {
            "alternatives": [
                {"transcript": "This is the first sentence.", "confidence": 0.98},
                {"transcript": "Here is another option.", "confidence": 0.90}
            ]
        },
        {
            "alternatives": [
                {"transcript": "And the second utterance.", "confidence": 0.97}
            ]
        },
        { # A result without alternatives (should be skipped)
            "languageCode": "en-US"
        }
    ]
}

extracted_transcripts = speech_converter.v2_json_to_dict(sample_v2_json)

print("--- Extracted Transcripts from Speech v2 JSON ---")
print(json.dumps(extracted_transcripts, indent=2))


## Conclusion

This tutorial has walked you through the features of the `format.py` module, demonstrating how to use its `Insights` class for converting various transcript formats to Google Contact Center AI Insights compatible JSON, and its `Speech` class for handling Google Cloud Speech-to-Text API responses. While the `Dlp` class is still under development, the existing functionalities provide powerful tools for data preparation and standardization.

Remember to replace placeholder credentials and mock data with your actual inputs when integrating this module into your production workflows.